# 📐 PhD Pure Mathematics — Topology, Differential Geometry & Proof Synthesis
### **1-Click Interactive Evaluation on Free Google Colab T4 GPU**
**Author:** Shreyansh Singh · [ExperimentLab.in](https://experimentlab.in) · [Hugging Face](https://huggingface.co/shreyansh12183/olmo2-7b-phd-pure-math)

---
### 🟢 How to Run This (1-Click / Zero Setup):
1. In the Colab top menu, click **Runtime** ➔ **Run all** (or press `Ctrl + F9`).
2. Wait ~2 minutes for the base model to load in 4-bit and attach the fine-tuned LoRA weights.
3. An interactive chat window will open below, with clickable example questions and a **live public URL** (`https://xxxx.gradio.live`) to test from any device!

In [ ]:
# STEP 1: Install High-Performance Inference Dependencies (~45s)
!pip install -q --upgrade pip
!pip install -q torch transformers peft bitsandbytes accelerate gradio

In [ ]:
# STEP 2: Load Model in 4-bit NF4 Quantization
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL_ID = "allenai/OLMo-2-1124-7B-Instruct"
ADAPTER_ID = "shreyansh12183/olmo2-7b-phd-pure-math"

print("📥 [1/3] Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)

print("⚡ [2/3] Loading Base Model (allenai/OLMo-2-1124-7B-Instruct) in 4-bit NF4...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

print("🎯 [3/3] Attaching LoRA Adapter (shreyansh12183/olmo2-7b-phd-pure-math)...")
model = PeftModel.from_pretrained(base_model, ADAPTER_ID)
model.eval()
print("\n✅ SUCCESS: Model is loaded and ready for live chat!")

In [ ]:
# STEP 3: Launch Native Zero-Error Multi-Turn Gradio Chat Interface
import gradio as gr

SYSTEM_PROMPT = """You are a PhD-level Pure Mathematics research model. You write rigorous, step-by-step mathematical proofs across Algebraic Topology, Differential Geometry, Real & Complex Analysis, and Abstract Algebra."""

def extract_text(val):
    if isinstance(val, str):
        return val
    if isinstance(val, dict):
        return val.get("content", "") or val.get("text", "")
    if hasattr(val, "content"):
        return str(val.content)
    return str(val) if val is not None else ""

def predict(message, history):
    user_text = extract_text(message).strip()
    if not user_text:
        return "Please enter a message."
    
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    
    # Robust multi-turn history extraction (supports Gradio 4, 5, 6, tuples, dicts, & ChatMessage objects)
    if history:
        recent_history = history[-10:] if len(history) > 10 else history
        for entry in recent_history:
            if isinstance(entry, dict):
                r = entry.get("role", "user")
                c = extract_text(entry.get("content", ""))
                if c:
                    messages.append({"role": r, "content": c})
            elif hasattr(entry, "role") and hasattr(entry, "content"):
                messages.append({"role": str(entry.role), "content": extract_text(entry.content)})
            elif isinstance(entry, (list, tuple)):
                if len(entry) >= 2:
                    u = extract_text(entry[0]).strip()
                    a = extract_text(entry[1]).strip()
                    if u:
                        messages.append({"role": "user", "content": u})
                    if a:
                        messages.append({"role": "assistant", "content": a})
                elif len(entry) == 1:
                    u = extract_text(entry[0]).strip()
                    if u:
                        messages.append({"role": "user", "content": u})
    
    messages.append({"role": "user", "content": user_text})
    
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=768,
            temperature=0.2,
            top_p=0.9,
            repetition_penalty=1.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
        
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return response.strip()

demo = gr.ChatInterface(
    fn=predict,
    title="📐 PhD Pure Mathematics — Topology, Differential Geometry & Proof Synthesis",
    description="1-Click Google Colab Evaluation | Model: shreyansh12183/olmo2-7b-phd-pure-math | ExperimentLab.in",
    examples=["Prove that the fundamental group of the torus T^2 is isomorphic to Z x Z.", "State and outline the proof of Sylow's Third Theorem in finite group theory.", "Derive the Bianchi identities for the Riemann Curvature Tensor in Riemannian geometry."]
)

demo.launch(share=True, debug=True)
